# 2.3 怎麼混合上下文特徵？

# 第 2 章：讓模型看更長的上下文

前置：第1章的索引、loss、梯度。這章只增加固定上下文，沒有 attention。向量是可以學習的數字列表，不是先驗語意標籤。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：上下文像把幾張卡片一起攤開**

拼接順序保留位置；窗口固定長度。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/context.svg")))

**先想一想：**輸入3維、輸出2維，weight 是3×2還是2×3？

一個輸出 feature 是輸入各 feature 的加權和加偏置。Linear 可以對非方陣輸入運作；用非方陣最能暴露 transpose 寫反。

**把直覺寫成數學：**$y=xW^T+b$；PyTorch 的 weight 是 [out,in]，x 是 [B,in]。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
layer = nn.Linear(3, 2)
x = torch.tensor([[1.0, 2.0, 4.0]])
y = layer(x)
manual = x @ layer.weight.T + layer.bias
print(layer.weight.shape, y)
assert torch.allclose(y, manual)

**如何讀結果：**矩陣乘法內側維度相等：1×3 乘 3×2 得到1×2。

**只改一件事：**只改輸出維度，先預測每個 shape。
